# **Step 1 : Dependencies Installation**

In [2]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


# **Step 2 : Model_Evalution** 

In [3]:
from ultralytics import YOLO

# Load model (explicit task)
model = YOLO(
    "/kaggle/input/10-epoch/pytorch/default/1/best.pt",
    task="detect"
)

# Run validation
metrics = model.val(
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    batch=16,
    device=0,          # single GPU only
    split="val",
    save_json=True,
    plots=True,
    verbose=True,
    visualize=True     # optional (slow)
)

# Print metrics
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Per-class mAP:", metrics.box.maps)

# Confusion matrix as DataFrame
df_cm = metrics.confusion_matrix.to_df()
print(df_cm)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,848,445 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 62.1±6.8 MB/s, size: 585.6 KB)
val: Scanning /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val/labels... 4196 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4196/4196 186.2it/s 22.5s<0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 

# **Step 3 : CSV File Creation**

In [4]:
import os

# Convert metrics to CSV string
val_csv = metrics.to_csv()
print(val_csv)

# Define output path
dir_path = "/kaggle/working/runs/detect"
csv_filename = "validation_results.csv"

# Ensure directory exists
os.makedirs(dir_path, exist_ok=True)

# Full file path
full_path = os.path.join(dir_path, csv_filename)

# Write CSV
with open(full_path, "w") as f:
    f.write(val_csv)

print("Saved at:", full_path)


Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
animal,219,753,0.61775,0.18061,0.2795,0.21672,0.09331
autorickshaw,1436,3205,0.7873,0.6271,0.69813,0.68894,0.48973
bicycle,264,301,0.66624,0.42445,0.51854,0.43742,0.25605
bus,1061,1794,0.79476,0.60814,0.68904,0.66634,0.50548
car,2582,8793,0.79227,0.57523,0.66652,0.6393,0.44769
caravan,18,18,0.42524,0.61675,0.5034,0.4377,0.40456
motorcycle,2778,10059,0.76216,0.59638,0.66916,0.64486,0.38222
person,2105,8863,0.75296,0.39231,0.51584,0.46644,0.25198
rider,2454,9444,0.75446,0.47808,0.58528,0.54864,0.31546
traffic light,157,370,0.7187,0.23478,0.35394,0.28687,0.14285
traffic sign,774,1404,0.61923,0.31268,0.41553,0.3264,0.17356
train,4,4,1.0,0.0,0.0,0.0,0.0
truck,1564,2758,0.74078,0.57252,0.64587,0.63955,0.45751
vehicle fallback,1116,2072,0.59533,0.09749,0.16754,0.13421,0.07282

Saved at: /kaggle/working/runs/detect/validation_results.csv


# **Step 4 : Model BenchMarking**

In [5]:
from ultralytics.utils.benchmarks import benchmark

benchmark(
    model="/kaggle/input/5-epoch/pytorch/default/1/best.pt",
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    half=False,
    device=0,
)


Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6636.8/8062.4 GB disk)

Benchmarks complete for /kaggle/input/5-epoch/pytorch/default/1/best.pt on /kaggle/input/kaggle-yaml/kaggle_new_data.yaml at imgsz=640 (198.77s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+---------------------------------------------------------------------------------------------------------+
|      Format                  Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS  |
+=========================================================================================================+
| 1    PyTorch                 ✅         49.6        0.1313                15.1                     66.2 |
| 2    TorchScript             ❌         0.0         -                     -                        -    |
| 3    ONNX                    ❌         0.0         -                     -                        -    |
| 4    OpenVINO                ❌         0.0         - 

,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""PyTorch""","""✅""","""49.6""","""0.1313""","""15.1""","""66.2"""
"""2""","""TorchScript""","""❌""","""0.0""","""-""","""-""","""-"""
"""3""","""ONNX""","""❌""","""0.0""","""-""","""-""","""-"""
"""4""","""OpenVINO""","""❌""","""0.0""","""-""","""-""","""-"""
"""5""","""TensorRT""","""❌""","""0.0""","""-""","""-""","""-"""
"""6""","""CoreML""","""❌""","""0.0""","""-""","""-""","""-"""
"""7""","""TensorFlow SavedModel""","""❌""","""0.0""","""-""","""-""","""-"""
"""8""","""TensorFlow GraphDef""","""❌""","""0.0""","""-""","""-""","""-"""
"""9""","""TensorFlow Lite""","""❌""","""0.0""","""-""","""-""","""-"""
"""10""","""TensorFlow Edge TPU""","""❌""","""0.0""","""-""","""-""","""-"""


# **Step 5 : Zip File Creation** 

In [6]:
import shutil
import os

folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/evaluation_results"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")

ZIP file created at: /kaggle/working/evaluation_results.zip


In [9]:
import os

file_path = "/kaggle/working/evaluation_results.zip"
print("Exists:", os.path.isfile(file_path))
print("Size (MB):", os.path.getsize(file_path) / (1024*1024))


Exists: True
Size (MB): 700.5066900253296


In [10]:
from IPython.display import FileLink

FileLink('/kaggle/working/evaluation_results.zip')


/kaggle/working/evaluation_results.zip

In [11]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/evaluation_results.zip"))


/kaggle/working/evaluation_results.zip

In [13]:
import os

file_path = "/kaggle/working/evaluation_results.zip"

if os.path.exists(file_path):
    print("✅ File exists:", file_path)
else:
    print("❌ File NOT found"ls
    !ls -lh /kaggle/working


✅ File exists: /kaggle/working/evaluation_results.zip
